# Log Analysis: 305440 — trader_1.py Live Performance

**Live result:** Total PnL = **7,656** (Pepper: 7,378 | Osmium: 278)  
**Backtest expectation:** Total ~84,257 per day  

This notebook pinpoints every factor limiting PnL.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

with open('LOGS_1/305440.log', 'r') as f:
    logdata = json.load(f)

with open('LOGS_1/305440.json', 'r') as f:
    jdata = json.load(f)

print(f"Status: {jdata['status']}")
print(f"Total profit: {jdata['profit']}")
print(f"End positions: {jdata['positions']}")

## 1. Parse Activities (Order Book Snapshots) and Trades

In [ ]:
from io import StringIO

activities = pd.read_csv(StringIO(logdata['activitiesLog']), sep=';')
print(f"Activities: {activities.shape}")
print(f"Days: {sorted(activities['day'].unique())}")
print(f"Timestamps: {activities['timestamp'].nunique()} unique")
print(f"Timestamp range: {activities['timestamp'].min()} - {activities['timestamp'].max()}")
print(f"Products: {sorted(activities['product'].unique())}")

os_act = activities[activities['product'] == 'ASH_COATED_OSMIUM'].copy()
pp_act = activities[activities['product'] == 'INTARIAN_PEPPER_ROOT'].copy()

os_act['mid'] = (os_act['bid_price_1'] + os_act['ask_price_1']) / 2
os_act['spread'] = os_act['ask_price_1'] - os_act['bid_price_1']
pp_act['mid'] = (pp_act['bid_price_1'] + pp_act['ask_price_1']) / 2

trades = pd.DataFrame(logdata['tradeHistory'])
print(f"\nTrades: {len(trades)} total")

## 2. CRITICAL FINDING: Only 1000 ticks / 100 timestamps

The live server runs **1000 timestamps** per day (0 to 99900, step 100).  
Our backtest CSVs have **10,000 timestamps** (0 to 999900, step 100).  
So the live day is **10x shorter** than our backtest day.  
This alone explains why live PnL ≈ backtest PnL / 10.

In [ ]:
live_ticks = os_act['timestamp'].nunique()
backtest_ticks = 10000
print(f"Live timestamps: {live_ticks}")
print(f"Backtest timestamps: {backtest_ticks}")
print(f"Ratio: {backtest_ticks / live_ticks:.1f}x")
print()
print(f"Expected PnL if linear scaling: {84257 / (backtest_ticks / live_ticks):.0f}")
print(f"Actual live PnL: {jdata['profit']:.0f}")

## 3. Osmium: Trade-by-Trade Analysis

In [ ]:
os_trades = trades[trades['symbol'] == 'ASH_COATED_OSMIUM'].copy()
our_os_buys = os_trades[os_trades['buyer'] == 'SUBMISSION'].copy()
our_os_sells = os_trades[os_trades['seller'] == 'SUBMISSION'].copy()

print("=== OUR OSMIUM BUYS ===")
if len(our_os_buys):
    print(our_os_buys[['timestamp', 'price', 'quantity']].to_string(index=False))
    print(f"Total: {our_os_buys['quantity'].sum()} units, avg price: {our_os_buys['price'].mean():.1f}")
else:
    print("None")

print("\n=== OUR OSMIUM SELLS ===")
if len(our_os_sells):
    print(our_os_sells[['timestamp', 'price', 'quantity']].to_string(index=False))
    print(f"Total: {our_os_sells['quantity'].sum()} units, avg price: {our_os_sells['price'].mean():.1f}")
else:
    print("None")

net_bought = our_os_buys['quantity'].sum() if len(our_os_buys) else 0
net_sold = our_os_sells['quantity'].sum() if len(our_os_sells) else 0
print(f"\nNet position: {net_bought - net_sold} (bought {net_bought}, sold {net_sold})")

# P&L from completed round-trips
sell_revenue = (our_os_sells['price'] * our_os_sells['quantity']).sum() if len(our_os_sells) else 0
buy_cost = (our_os_buys['price'] * our_os_buys['quantity']).sum() if len(our_os_buys) else 0
print(f"Sell revenue: {sell_revenue:.0f}")
print(f"Buy cost: {buy_cost:.0f}")
print(f"Cash P&L (ignoring end position): {sell_revenue - buy_cost:.0f}")

## 4. PROBLEM #1: Osmium — Massive One-Sided Position (-76)

We sold 101 units but only bought back 25. End position = **-76 short**.  
The strategy is selling into bids above 10001 (Phase 2 entries) but  
**never gets to buy back** because the flatten phase needs `ask <= 10000`,  
and the live book rarely has ask that low.

In [ ]:
# How often is ask <= 10000 (our exit threshold for shorts)?
ask_at_fair = (os_act['ask_price_1'] <= 10000).sum()
total = len(os_act)
print(f"Ticks with ask <= 10000 (can cover shorts): {ask_at_fair}/{total} = {ask_at_fair/total:.1%}")
print(f"Ticks with ask <= 9999 (cheap buys): {(os_act['ask_price_1'] <= 9999).sum()}/{total}")
print()

# How often is bid >= 10001 (our entry for shorts)?
bid_rich = (os_act['bid_price_1'] >= 10001).sum()
print(f"Ticks with bid >= 10001 (can sell short): {bid_rich}/{total} = {bid_rich/total:.1%}")
print(f"Ticks with bid >= 10003 (edge 3+ sells): {(os_act['bid_price_1'] >= 10003).sum()}/{total}")
print()

print("=== The asymmetry: lots of sell opportunities, very few buy-back opportunities ===")
print(f"Ask price distribution:")
print(os_act['ask_price_1'].describe())
print(f"\nBid price distribution:")
print(os_act['bid_price_1'].describe())

In [ ]:
# Visualize Osmium mid, our trades, and position buildup
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

# Mid price
axes[0].plot(os_act['timestamp'], os_act['mid'], 'b-', alpha=0.7, linewidth=0.8)
axes[0].axhline(y=10000, color='gray', linestyle='--', alpha=0.5, label='Fair (10000)')

# Our buys (green) and sells (red)
if len(our_os_buys):
    axes[0].scatter(our_os_buys['timestamp'], our_os_buys['price'],
                    c='green', s=our_os_buys['quantity']*10, zorder=5, label='Our buys')
if len(our_os_sells):
    axes[0].scatter(our_os_sells['timestamp'], our_os_sells['price'],
                    c='red', s=our_os_sells['quantity']*10, zorder=5, label='Our sells')

axes[0].set_ylabel('Price')
axes[0].set_title('Osmium: Mid Price + Our Trades')
axes[0].legend()

# Position over time (reconstructed from trades)
all_our_os = pd.concat([
    our_os_buys[['timestamp','quantity']].assign(side=1),
    our_os_sells[['timestamp','quantity']].assign(side=-1)
]).sort_values('timestamp')
all_our_os['delta'] = all_our_os['quantity'] * all_our_os['side']
all_our_os['position'] = all_our_os['delta'].cumsum()
axes[1].step(all_our_os['timestamp'], all_our_os['position'], 'purple', where='post')
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_ylabel('Position')
axes[1].set_title('Osmium: Our Position Over Time')

# PnL
axes[2].plot(os_act['timestamp'], os_act['profit_and_loss'], 'darkorange', linewidth=1)
axes[2].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[2].set_ylabel('PnL')
axes[2].set_title('Osmium: Mark-to-Market PnL')
axes[2].set_xlabel('Timestamp')

plt.tight_layout()
plt.show()

## 5. PROBLEM #2: Osmium — Sell at Edge 1-2 Builds Up Net Short

In the early ticks (0-2300), we sell 56 units at prices 10001-10010.  
These are mostly edge 1-3 sells. But we **never buy back** because:  
- `EXIT_BUY_MAX = 10000` — we need ask ≤ 10000 to cover
- The ask stays at 10004-10015 most of the time
- Only at ts=32800 and 44800 do we finally get asks at 9998-9999

**Root cause:** The sell side works perfectly (edge-aware shorts).  
The buy-back side is broken in live — ask rarely comes to 10000.

In [ ]:
# Timeline: when did we build the short position?
print("Position buildup timeline:")
print(all_our_os[['timestamp', 'side', 'quantity', 'position']].to_string(index=False))
print()

# What was the book like at each sell?
print("\nBook state at each of our sells:")
for _, trade in our_os_sells.iterrows():
    ts = trade['timestamp']
    book = os_act[os_act['timestamp'] == ts]
    if len(book):
        b = book.iloc[0]
        print(f"  ts={ts:>6}  sold {trade['quantity']:>2} @ {trade['price']:>6.0f}  "
              f"book: bid1={b['bid_price_1']:.0f} ask1={b['ask_price_1']:.0f} "
              f"mid={b['mid']:.1f} spread={b['spread']:.0f}")

## 6. PROBLEM #3: Pepper — Only 7,378 from 80 Lots

Backtest expects ~79,364 per day. Live gets 7,378. Why?

In [ ]:
pp_trades_df = trades[trades['symbol'] == 'INTARIAN_PEPPER_ROOT'].copy()
pp_buys = pp_trades_df[pp_trades_df['buyer'] == 'SUBMISSION']
pp_sells = pp_trades_df[pp_trades_df['seller'] == 'SUBMISSION']

print("=== PEPPER BUYS ===")
print(pp_buys[['timestamp', 'price', 'quantity']].to_string(index=False))
print(f"Total bought: {pp_buys['quantity'].sum()} units")
print(f"Avg buy price: {pp_buys['price'].mean():.1f}")
print()

print("=== PEPPER SELLS ===")
if len(pp_sells):
    print(pp_sells[['timestamp', 'price', 'quantity']].to_string(index=False))
else:
    print("None — we never sold!")
print()

# Pepper price evolution
first_mid = pp_act['mid'].iloc[0]
last_mid = pp_act['mid'].iloc[-1]
first_ts = pp_act['timestamp'].iloc[0]
last_ts = pp_act['timestamp'].iloc[-1]
total_drift = last_mid - first_mid
print(f"Pepper mid: {first_mid:.1f} -> {last_mid:.1f} (drift: +{total_drift:.1f})")
print(f"Timestamp range: {first_ts} -> {last_ts} ({last_ts - first_ts} ticks)")
print(f"Slope: {total_drift / (last_ts - first_ts):.6f} per tick")
print()

# Our buy cost vs mark-to-market
total_cost = (pp_buys['price'] * pp_buys['quantity']).sum()
end_value = 80 * last_mid
print(f"Total cost: {total_cost:.0f}")
print(f"End value (80 * {last_mid:.0f}): {end_value:.0f}")
print(f"Unrealized PnL: {end_value - total_cost:.0f}")
print(f"Reported PnL: {pp_act['profit_and_loss'].iloc[-1]:.0f}")

In [ ]:
# Pepper price chart
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

axes[0].plot(pp_act['timestamp'], pp_act['mid'], 'b-', linewidth=0.8)
if len(pp_buys):
    axes[0].scatter(pp_buys['timestamp'], pp_buys['price'],
                    c='green', s=pp_buys['quantity']*5, zorder=5, label='Our buys')
axes[0].set_ylabel('Price')
axes[0].set_title('Pepper: Mid Price + Our Buys')
axes[0].legend()

axes[1].plot(pp_act['timestamp'], pp_act['profit_and_loss'], 'darkorange', linewidth=1)
axes[1].set_ylabel('PnL')
axes[1].set_title('Pepper: PnL')
axes[1].set_xlabel('Timestamp')

plt.tight_layout()
plt.show()

## 7. PROBLEM #4: Pepper PnL = drift * position * time

Pepper is a drift strategy. PnL = position × price change.  

**Backtest:** 10,000 ticks × 100 step = 999,900 timestamp range. Drift = 0.001 × 999,900 ≈ 1000 points. 80 lots × 1000 = 80,000 PnL.  
**Live:** 1,000 ticks × 100 step = 99,900 timestamp range. Drift = 0.001 × 99,900 ≈ 100 points. 80 lots × 100 = 8,000 PnL.  

**Actual live Pepper PnL = 7,378** — close to the 8,000 estimate.  
The gap is entry cost (we bought at 13007 vs fair ~13000, losing ~7 × 80 = 560).

In [ ]:
# Verify the math
live_ts_range = pp_act['timestamp'].max() - pp_act['timestamp'].min()
expected_drift = 0.001 * live_ts_range
expected_pnl = 80 * expected_drift
entry_cost = (pp_buys['price'] * pp_buys['quantity']).sum() / pp_buys['quantity'].sum() - first_mid
expected_pnl_adj = expected_pnl - entry_cost * 80

print(f"Live timestamp range: {live_ts_range}")
print(f"Expected drift: {expected_drift:.1f} points")
print(f"Expected PnL (80 * drift): {expected_pnl:.0f}")
print(f"Entry cost above first mid: {entry_cost:.1f}/unit")
print(f"Adjusted expected PnL: {expected_pnl_adj:.0f}")
print(f"Actual Pepper PnL: {pp_act['profit_and_loss'].iloc[-1]:.0f}")

## 8. Osmium: Missed Opportunities Analysis

In [ ]:
# For each tick, what could we have done?
# Count ticks with actionable levels
os_act_clean = os_act.dropna(subset=['bid_price_1', 'ask_price_1']).copy()

buy_opp = os_act_clean[os_act_clean['ask_price_1'] <= 9999]
sell_opp = os_act_clean[os_act_clean['bid_price_1'] >= 10001]
cover_opp = os_act_clean[os_act_clean['ask_price_1'] <= 10000]
flatten_opp = os_act_clean[os_act_clean['bid_price_1'] >= 10000]

print(f"Out of {len(os_act_clean)} ticks:")
print(f"  Buy opportunities (ask <= 9999):  {len(buy_opp):>4} ({len(buy_opp)/len(os_act_clean):.1%})")
print(f"  Sell opportunities (bid >= 10001): {len(sell_opp):>4} ({len(sell_opp)/len(os_act_clean):.1%})")
print(f"  Cover short (ask <= 10000):        {len(cover_opp):>4} ({len(cover_opp)/len(os_act_clean):.1%})")
print(f"  Flatten long (bid >= 10000):       {len(flatten_opp):>4} ({len(flatten_opp)/len(os_act_clean):.1%})")
print()

# Distribution of ask and bid prices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(os_act_clean['bid_price_1'], bins=30, color='blue', alpha=0.7, edgecolor='black')
axes[0].axvline(x=10000, color='red', linestyle='--', label='Fair 10000')
axes[0].axvline(x=10001, color='green', linestyle='--', label='Sell threshold 10001')
axes[0].set_title('Osmium Bid Price Distribution')
axes[0].legend()

axes[1].hist(os_act_clean['ask_price_1'], bins=30, color='orange', alpha=0.7, edgecolor='black')
axes[1].axvline(x=10000, color='red', linestyle='--', label='Fair 10000')
axes[1].axvline(x=9999, color='green', linestyle='--', label='Buy threshold 9999')
axes[1].set_title('Osmium Ask Price Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. ROOT CAUSE SUMMARY

### Issue 1: Live day is 10x shorter than backtest (dominant factor)
- Backtest: 10,000 timestamps (0-999,900)
- Live: 1,000 timestamps (0-99,900)
- This alone means live PnL ≈ backtest / 10
- Pepper: 79,364 / 10 ≈ 7,936 vs actual 7,378 ✓

### Issue 2: Osmium bleeds from one-sided inventory
- Sold 101 units, bought back only 25 → stuck at -76 short
- Sells at 10001-10010 (good entries)
- But can't buy back: ask rarely reaches 10000
- End position -76 × (mid - 10000) = unrealized loss eating cash profits

### Issue 3: Asymmetric book in live
- Bid is more often above 10001 (sell opportunities frequent)
- Ask almost never at 9999 or below (buy opportunities rare)
- Our exit threshold EXIT_BUY_MAX = 10000 is too tight for live conditions

### Issue 4: No maker quotes
- trader_final had maker quotes (passive orders inside spread)
- trader_1 removed them for "purer" mean-reversion
- In live, maker fills DO happen → trader_final's maker quotes would add PnL

In [ ]:
# Quantify each issue
print("=== PnL Attribution ===")
print()

# Osmium cash P&L
os_sell_rev = (our_os_sells['price'] * our_os_sells['quantity']).sum() if len(our_os_sells) else 0
os_buy_cost = (our_os_buys['price'] * our_os_buys['quantity']).sum() if len(our_os_buys) else 0
os_cash_pnl = os_sell_rev - os_buy_cost
os_end_pos = net_bought - net_sold
os_end_mid = os_act['mid'].iloc[-1]
os_unrealized = os_end_pos * os_end_mid  # negative pos * mid = negative value
os_total_pnl = os_cash_pnl + os_unrealized

net_bought = our_os_buys['quantity'].sum() if len(our_os_buys) else 0
net_sold = our_os_sells['quantity'].sum() if len(our_os_sells) else 0

print(f"OSMIUM:")
print(f"  Cash from sells: +{os_sell_rev:.0f} ({net_sold} units, avg {os_sell_rev/net_sold:.0f})")
print(f"  Cash from buys:  -{os_buy_cost:.0f} ({net_bought} units, avg {os_buy_cost/net_bought:.0f})")
print(f"  Net cash: {os_cash_pnl:.0f}")
print(f"  End position: {net_bought - net_sold} @ mid={os_end_mid:.0f}")
print(f"  Unrealized: {(net_bought - net_sold) * os_end_mid:.0f}")
print(f"  Reported PnL: {os_act['profit_and_loss'].iloc[-1]:.0f}")
print()
print(f"PEPPER:")
print(f"  Bought 80 lots early, held for drift")
print(f"  Reported PnL: {pp_act['profit_and_loss'].iloc[-1]:.0f}")
print()
print(f"TOTAL: {jdata['profit']:.0f}")

## 10. Recommendations for trader_1 Improvements

### A. Widen Osmium exit thresholds
- `EXIT_BUY_MAX = 10000` is too tight — ask almost never reaches 10000 in live
- Consider `EXIT_BUY_MAX = 10002` (accept 2-tick loss on exit to free position)
- Same for `EXIT_SELL_MIN`: consider lowering to 9998

### B. Add maker quotes back
- trader_final placed passive buy/sell quotes inside the spread
- In live, these DO get filled (unlike taker-only backtest)
- Adding makers at e.g. bid+1 / ask-1 would capture spread and help flatten

### C. Reduce Osmium position aggressiveness
- TIER_1_CAP=10 at edge 1 still leads to -76 buildup
- Consider TIER_1_CAP=5 to limit exposure at thin edges
- Or skip edge-1 entirely and only trade edge 2+

### D. Pepper is working correctly
- PnL scales linearly with time; live day is just shorter
- Entry cost (~7/unit) is acceptable given the 100-point drift

In [ ]:
# Combined PnL chart from graphLog
graph = pd.read_csv(StringIO(jdata['graphLog']), sep=';')

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(graph['timestamp'], graph['value'], 'darkblue', linewidth=1)
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Timestamp')
ax.set_ylabel('Total PnL')
ax.set_title(f'Combined PnL Over Time (Final: {jdata["profit"]:.0f})')
ax.fill_between(graph['timestamp'], graph['value'], alpha=0.15, color='blue')
plt.tight_layout()
plt.show()

print(f"PnL went negative until ~ts={graph[graph['value'] >= 0]['timestamp'].iloc[0]}")
print(f"Min PnL: {graph['value'].min():.0f} at ts={graph.loc[graph['value'].idxmin(), 'timestamp']}")
print(f"Max PnL: {graph['value'].max():.0f} at ts={graph.loc[graph['value'].idxmax(), 'timestamp']}")

In [ ]:
# Market trades (other participants) — what is the "true" fair value?
market_os = os_trades[(os_trades['buyer'] != 'SUBMISSION') & (os_trades['seller'] != 'SUBMISSION')]

print("Market Osmium trades (not ours):")
print(f"  Count: {len(market_os)}")
print(f"  Price range: {market_os['price'].min():.0f} - {market_os['price'].max():.0f}")
print(f"  Mean: {market_os['price'].mean():.1f}")
print(f"  Median: {market_os['price'].median():.1f}")
print()

# If fair is actually ~10005 in live (not 10000), our whole strategy is mis-calibrated
print(f"Live Osmium mid mean: {os_act['mid'].mean():.1f}")
print(f"Live Osmium mid median: {os_act['mid'].median():.1f}")
print(f"Our assumed fair: 10000")
print(f"Difference: {os_act['mid'].mean() - 10000:.1f} (market trades higher than our fair!)")

## 11. KEY INSIGHT: Is Fair Value Actually 10000 in Live?

If the live market mid is consistently above 10000, then our `OSMIUM_FAIR = 10000` 
is **too low**, making us sell (short) constantly and never covering. This is the 
single most important issue. 

**Fix:** Use a rolling/adaptive fair value instead of hardcoded 10000, or adjust 
OSMIUM_FAIR upward.

In [ ]:
# Final diagnostic: what fair value would have been optimal?
# If we had set fair = X, we'd sell when bid > X and buy when ask < X
# The market mid fluctuates around some value — let's find it

mid_values = os_act['mid'].dropna()
print(f"Mid statistics:")
print(f"  Mean: {mid_values.mean():.2f}")
print(f"  Std: {mid_values.std():.2f}")
print(f"  5th percentile: {mid_values.quantile(0.05):.1f}")
print(f"  95th percentile: {mid_values.quantile(0.95):.1f}")
print()

# Mid over time with rolling mean
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(os_act['timestamp'], os_act['mid'], 'b-', alpha=0.5, linewidth=0.5, label='Mid')
rolling = os_act.set_index('timestamp')['mid'].rolling(50, min_periods=1).mean()
ax.plot(rolling.index, rolling.values, 'red', linewidth=1.5, label='Rolling mean (50)')
ax.axhline(y=10000, color='gray', linestyle='--', alpha=0.7, label='Our fair (10000)')
ax.axhline(y=mid_values.mean(), color='green', linestyle='--', alpha=0.7, label=f'Actual mean ({mid_values.mean():.0f})')
ax.set_title('Osmium: Is 10000 the Right Fair Value?')
ax.set_ylabel('Price')
ax.set_xlabel('Timestamp')
ax.legend()
plt.tight_layout()
plt.show()